# Titanic Survival Prediction
### Exploratory Data Analysis, Feature Engineering, and Classification Model Evaluation

## Project overview

This project investigates passenger survival in the Kaggle Titanic dataset through exploratory data analysis and supervised machine learning. The analysis examines the structure and quality of the data, identifies descriptive relationships between passenger characteristics and survival, and evaluates a set of classification algorithms using a controlled model-selection workflow.

The objective is to build a reproducible analytical comparison rather than optimize for a competition leaderboard. The workflow combines interpretable feature engineering, pipeline-based preprocessing, stratified cross-validation, validation-based model selection, and a final evaluation on an independent test partition.

## Analytical workflow

1. Inspect the dataset structure, variable types, missing values, and target distribution.
2. Examine survival patterns across selected passenger characteristics.
3. Construct interpretable features from the available passenger information.
4. Partition the labelled data into training, validation, and test sets.
5. Apply preprocessing within model pipelines to prevent information leakage.
6. Tune candidate classifiers using stratified five-fold cross-validation.
7. Select a model using validation performance and report its final test performance.

**Reproducibility and data scope:** The analysis uses the labelled `train.csv` file from the Kaggle Titanic competition because it contains the `Survived` target. The competition's unlabelled `test.csv` is not used for model evaluation. Randomized operations use a fixed seed, and preprocessing parameters are learned within the relevant training folds or partition.

## 1. Dataset and experimental setup

The analysis uses the labelled passenger dataset from the [Kaggle Titanic competition](https://www.kaggle.com/competitions/titanic/data). Each observation represents a passenger, while the available variables describe passenger attributes, ticket and cabin information, family relationships, and survival status.

The labelled training file, `train.csv`, is the source for exploratory analysis and model development. The unlabelled competition test file is excluded because it does not provide ground-truth survival labels for evaluation.

For reproducibility, the notebook searches a small set of conventional local dataset locations. The dataset itself is treated as an external input rather than a committed project artifact. A fixed random seed is used for data partitioning, cross-validation, and stochastic estimators. Numerical results may vary slightly across software versions and execution environments.

In [ ]:
from pathlib import Path
import warnings
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, ConfusionMatrixDisplay,
    confusion_matrix
)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 120)
sns.set_theme(style="whitegrid", context="notebook")

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)

## 2. Data loading

The labelled Titanic dataset is loaded from the configured data location. The subsequent analysis uses this single source table for data inspection, exploratory analysis, feature construction, and the stratified training, validation, and test partitions.

In [ ]:
candidate_paths = [
    Path("data/raw/train.csv"),
    Path("/kaggle/input/titanic/train.csv"),
    Path("/content/drive/MyDrive/Data Science Project/train.csv"),
]

DATA_PATH = next((path for path in candidate_paths if path.exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find train.csv. Download it from Kaggle and place it at "
        "'data/raw/train.csv', or update DATA_PATH."
    )

df = pd.read_csv(DATA_PATH)
print(f"Loaded: {DATA_PATH}")
print("Shape:", df.shape)
display(df.head())

## 3. Dataset structure and data quality

The initial inspection examines the dataset dimensions, column names, data types, descriptive statistics, and missing-value distribution. These checks establish the analytical structure of the data and identify variables that require preprocessing before model fitting.

In [ ]:
display(df.info())
display(df.describe(include="all").T)
missing = (
    df.isna().sum()
      .to_frame("missing_count")
      .assign(missing_percent=lambda x: 100 * x["missing_count"] / len(df))
      .sort_values("missing_percent", ascending=False)
)
display(missing[missing["missing_count"] > 0])

### 3.1 Survival target distribution

`Survived` is the binary prediction target: `0` indicates that a passenger did not survive, while `1` indicates that a passenger survived. The class distribution provides the baseline context for model evaluation and helps determine whether accuracy should be interpreted alongside class-sensitive metrics.

In [ ]:
target_counts = df["Survived"].value_counts().sort_index()
display(pd.DataFrame({
    "count": target_counts,
    "proportion": df["Survived"].value_counts(normalize=True).sort_index()
}))
ax = sns.countplot(data=df, x="Survived")
ax.set(title="Passenger survival counts", xlabel="Survived (0 = No, 1 = Yes)", ylabel="Passengers")
plt.show()

## 4. Exploratory data analysis

Exploratory analysis examines how survival outcomes vary across passenger characteristics, including sex, passenger class, age, fare, and family composition. These comparisons are descriptive and are used to identify patterns in the observed dataset. They do not establish causal relationships.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.barplot(data=df, x="Sex", y="Survived", errorbar=None, ax=axes[0])
axes[0].set_title("Survival rate by sex")
sns.barplot(data=df, x="Pclass", y="Survived", errorbar=None, ax=axes[1])
axes[1].set_title("Survival rate by passenger class")
sns.barplot(data=df, x="Embarked", y="Survived", errorbar=None, ax=axes[2])
axes[2].set_title("Survival rate by embarkation port")
for ax in axes:
    ax.set_ylabel("Mean survival rate")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(data=df, x="Age", hue="Survived", bins=25, kde=True,
             element="step", stat="density", common_norm=False, ax=axes[0])
axes[0].set_title("Age distribution by survival")
sns.boxplot(data=df, x="Survived", y="Fare", ax=axes[1])
axes[1].set_title("Fare distribution by survival")
plt.tight_layout()
plt.show()

In [ ]:
# Survival rates by family-related variables
eda = df.copy()
eda["Family_Size"] = eda["SibSp"] + eda["Parch"] + 1
eda["Is_Alone"] = (eda["Family_Size"] == 1).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.barplot(data=eda, x="Family_Size", y="Survived", errorbar=None, ax=axes[0])
axes[0].set_title("Survival rate by family size")
sns.barplot(data=eda, x="Is_Alone", y="Survived", errorbar=None, ax=axes[1])
axes[1].set_title("Survival rate: travelling alone")
axes[1].set_xticks([0, 1], ["With family", "Alone"])
plt.tight_layout()
plt.show()

### EDA observations

The visualizations and grouped summaries provide a descriptive comparison of survival outcomes across the selected passenger variables. The results are interpreted in the context of the observed sample and the distribution of each variable. These associations inform the subsequent feature-engineering and modelling stages; they are not treated as evidence of causation.

In [ ]:
display(
    df.groupby("Sex", observed=True)["Survived"].agg(["count", "mean"])
      .rename(columns={"mean": "survival_rate"})
)
display(
    df.groupby("Pclass", observed=True)["Survived"].agg(["count", "mean"])
      .rename(columns={"mean": "survival_rate"})
)
display(
    eda.groupby("Family_Size", observed=True)["Survived"].agg(["count", "mean"])
      .rename(columns={"mean": "survival_rate"})
)

## 5. Feature engineering

A compact set of passenger-level features is derived from the original variables to represent family composition, name information, and ticket and cabin characteristics.

| Engineered feature | Definition |
|---|---|
| `Family_Size` | Total number of listed family members travelling with the passenger, including the passenger. |
| `Is_Alone` | Binary indicator of whether the passenger travelled without listed family members. |
| `Title` | Title extracted from the passenger's name, with infrequent titles grouped. |
| `Name_Length` | Number of characters in the passenger's name. |
| `Cabin_Deck` | First character of the cabin value; missing values are represented by an explicit unknown category. |
| `Cabin_Known` | Indicator of whether a cabin value was recorded. |
| `Ticket_Prefix` | Normalized non-numeric ticket prefix; numeric-only tickets are assigned `NUMERIC`. |

These transformations are based on the available passenger records and do not use the target variable to construct features. The resulting variables retain interpretable links to the original dataset.

In [ ]:
def add_features(data: pd.DataFrame) -> pd.DataFrame:
    out = data.copy()

    out["Family_Size"] = out["SibSp"] + out["Parch"] + 1
    out["Is_Alone"] = (out["Family_Size"] == 1).astype(int)

    # Extract the title between the comma and period in the passenger name.
    title = out["Name"].str.extract(r",\\s*([^.]*)\\.", expand=False).str.strip()
    title = title.replace({
        "Mlle": "Miss",
        "Ms": "Miss",
        "Mme": "Mrs",
        "Lady": "Rare",
        "Countess": "Rare",
        "Capt": "Rare",
        "Col": "Rare",
        "Don": "Rare",
        "Dr": "Rare",
        "Major": "Rare",
        "Rev": "Rare",
        "Sir": "Rare",
        "Jonkheer": "Rare",
        "Dona": "Rare",
    })
    common_titles = {"Mr", "Mrs", "Miss", "Master"}
    out["Title"] = title.where(title.isin(common_titles), "Rare").fillna("Unknown")

    out["Name_Length"] = out["Name"].fillna("").str.len()

    cabin = out["Cabin"].fillna("")
    out["Cabin_Known"] = cabin.ne("").astype(int)
    out["Cabin_Deck"] = cabin.str[0].replace("", "Unknown")

    # Remove digits and punctuation; normalize whitespace and case.
    ticket_prefix = (
        out["Ticket"].fillna("UNKNOWN")
        .str.replace(r"[0-9]", "", regex=True)
        .str.replace(r"[./]", "", regex=True)
        .str.replace(r"\\s+", "", regex=True)
        .str.upper()
    )
    out["Ticket_Prefix"] = ticket_prefix.replace("", "NUMERIC")

    return out

featured = add_features(df)
display(featured[[
    "Name", "Title", "Family_Size", "Is_Alone", "Name_Length",
    "Cabin", "Cabin_Deck", "Cabin_Known", "Ticket", "Ticket_Prefix"
]].head())

## 6. Predictor definition and data partitioning

`Survived` is retained as the target, while passenger identifiers and high-cardinality raw text fields are excluded from the predictor matrix after the relevant information has been represented through engineered features.

The labelled data is divided into three stratified partitions:

- **Training set (60%)** — used to fit candidate models and tune hyperparameters through cross-validation.
- **Validation set (20%)** — used to compare tuned candidates and select the final model.
- **Test set (20%)** — reserved for a one-time estimate of generalization performance after model selection.

Stratification preserves the approximate target-class proportions across partitions. The test partition is excluded from feature decisions, hyperparameter tuning, and model selection.

In [ ]:
TARGET = "Survived"
DROP_COLUMNS = ["PassengerId", "Name", "Ticket", "Cabin", TARGET]

X = featured.drop(columns=DROP_COLUMNS)
y = featured[TARGET].astype(int)

# First reserve 20% for the final test set.
X_train_valid, X_test, y_train_valid, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=SEED,
    stratify=y
)

# Split the remaining 80% so validation is 20% of the full dataset.
# 0.25 x 0.80 = 0.20
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_valid, y_train_valid,
    test_size=0.25,
    random_state=SEED,
    stratify=y_train_valid
)

split_summary = pd.DataFrame({
    "partition": ["Train", "Validation", "Test"],
    "rows": [len(X_train), len(X_valid), len(X_test)],
    "share_of_dataset": [
        len(X_train) / len(X),
        len(X_valid) / len(X),
        len(X_test) / len(X)
    ],
    "survival_rate": [y_train.mean(), y_valid.mean(), y_test.mean()]
})
display(split_summary)

assert set(X_train.index).isdisjoint(X_valid.index)
assert set(X_train.index).isdisjoint(X_test.index)
assert set(X_valid.index).isdisjoint(X_test.index)

## 7. Preprocessing pipeline

Preprocessing is performed within scikit-learn pipelines so that learned transformations are fitted only on the training data available to each model-fitting step.

For numerical variables, missing values are imputed using the median and values are scaled. For categorical variables, missing values are imputed using the most frequent category and categories are one-hot encoded. Embedding these operations in the pipeline ensures that cross-validation folds do not use information from their validation folds when estimating imputation or scaling parameters.

Scaling supports distance- and margin-based methods, including K-Nearest Neighbors, Support Vector Machines, and Logistic Regression. Although tree-based models generally do not require feature scaling, the shared pipeline structure provides a consistent preprocessing interface for model comparison.

In [ ]:
numeric_features = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=["number"]).columns.tolist()

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features)
    ],
    remainder="drop"
)

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

## 8. Model selection and cross-validation

Six classification algorithms are included to compare distinct modelling approaches:

- **Logistic Regression** — linear classification baseline.
- **Decision Tree** — non-linear model with rule-based splits.
- **Random Forest** — ensemble of decision trees.
- **Support Vector Machine** — margin-based classification.
- **K-Nearest Neighbors** — distance-based classification.
- **Gaussian Naive Bayes** — probabilistic classifier based on conditional independence assumptions.

Hyperparameters are tuned on the training partition using stratified five-fold cross-validation. ROC-AUC is the primary tuning metric. Accuracy, precision, recall, and F1-score are also reported to provide a broader view of classification performance.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

model_grids = {
    "Logistic Regression": (
        LogisticRegression(max_iter=2000, random_state=SEED),
        {"model__C": [0.1, 1.0, 10.0]}
    ),
    "Decision Tree": (
        DecisionTreeClassifier(random_state=SEED),
        {
            "model__max_depth": [3, 5, 8, None],
            "model__min_samples_leaf": [1, 2, 5]
        }
    ),
    "Random Forest": (
        RandomForestClassifier(random_state=SEED, n_jobs=-1),
        {
            "model__n_estimators": [200, 400],
            "model__max_depth": [None, 5, 10],
            "model__min_samples_leaf": [1, 2, 4]
        }
    ),
    "SVM": (
        SVC(probability=True, random_state=SEED),
        {
            "model__C": [0.1, 1.0, 10.0],
            "model__kernel": ["linear", "rbf"]
        }
    ),
    "KNN": (
        KNeighborsClassifier(),
        {
            "model__n_neighbors": [5, 7, 11, 15],
            "model__weights": ["uniform", "distance"]
        }
    ),
    "Gaussian Naive Bayes": (
        GaussianNB(),
        {"model__var_smoothing": [1e-9, 1e-8, 1e-7]}
    )
}

searches = {}
best_estimators = {}
cv_rows = []

for name, (estimator, param_grid) in model_grids.items():
    pipeline = Pipeline(steps=[
        ("preprocessor", clone(preprocessor)),
        ("model", estimator)
    ])

    search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        scoring="roc_auc",
        cv=cv,
        n_jobs=-1,
        refit=True,
        return_train_score=False
    )
    search.fit(X_train, y_train)

    searches[name] = search
    best_estimators[name] = search.best_estimator_
    cv_rows.append({
        "Model": name,
        "Best CV ROC-AUC": search.best_score_,
        "Best parameters": search.best_params_
    })

cv_results = pd.DataFrame(cv_rows).sort_values("Best CV ROC-AUC", ascending=False)
display(cv_results.reset_index(drop=True))

## 9. Validation-set model comparison

Each tuned candidate is evaluated on the same validation partition, which remains separate from cross-validation and model fitting. Validation performance provides the basis for selecting the candidate that proceeds to final testing.

ROC-AUC measures the model's ability to rank positive cases above negative cases across classification thresholds. Accuracy reports the overall proportion of correct predictions, while precision and recall describe positive-class prediction quality. F1-score summarizes the balance between precision and recall at the model's default classification threshold. These metrics are considered together rather than relying on accuracy alone.

In [ ]:
def evaluate_classifier(model, X_part, y_part):
    predictions = model.predict(X_part)

    if hasattr(model, "predict_proba"):
        probabilities = model.predict_proba(X_part)[:, 1]
    elif hasattr(model, "decision_function"):
        probabilities = model.decision_function(X_part)
    else:
        probabilities = None

    return {
        "Accuracy": accuracy_score(y_part, predictions),
        "Precision": precision_score(y_part, predictions, zero_division=0),
        "Recall": recall_score(y_part, predictions, zero_division=0),
        "F1": f1_score(y_part, predictions, zero_division=0),
        "ROC-AUC": roc_auc_score(y_part, probabilities) if probabilities is not None else np.nan
    }

validation_rows = []
for name, model in best_estimators.items():
    metrics = evaluate_classifier(model, X_valid, y_valid)
    validation_rows.append({"Model": name, **metrics})

validation_results = (
    pd.DataFrame(validation_rows)
      .sort_values(["ROC-AUC", "F1"], ascending=False)
      .reset_index(drop=True)
)
display(validation_results.style.format({
    "Accuracy": "{:.3f}", "Precision": "{:.3f}", "Recall": "{:.3f}",
    "F1": "{:.3f}", "ROC-AUC": "{:.3f}"
}))

### 9.1 Model selection criterion

The final candidate is selected using the highest validation ROC-AUC. This criterion prioritizes discrimination across classification thresholds and provides an explicit, reproducible selection rule. The accompanying validation metrics provide additional context for interpreting the selected model's performance, including its precision, recall, F1-score, and overall accuracy.

In [ ]:
selected_name = validation_results.loc[0, "Model"]
selected_model = best_estimators[selected_name]

print("Selected model:", selected_name)
print("Validation metrics:")
display(validation_results.loc[validation_results["Model"] == selected_name])

## 10. Final test-set evaluation

Following validation-based model selection, the selected estimator is evaluated on the held-out test partition. This test set provides the final estimate of performance on observations that were not used for model fitting, hyperparameter tuning, or model selection.

The test results are reported once and are not used to revise features, tune hyperparameters, or choose a different estimator. The evaluation includes classification metrics and a confusion matrix to show the distribution of correct and incorrect predictions.

In [ ]:
test_metrics = evaluate_classifier(selected_model, X_test, y_test)
print("Final test metrics")
display(pd.DataFrame([test_metrics], index=[selected_name]).T.rename(columns={selected_name: "Test score"}))

test_predictions = selected_model.predict(X_test)
print(classification_report(
    y_test, test_predictions,
    target_names=["Did not survive", "Survived"],
    zero_division=0
))

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test, test_predictions,
    display_labels=["Did not survive", "Survived"],
    cmap="Blues",
    colorbar=False,
    ax=ax
)
ax.set_title(f"Test confusion matrix — {selected_name}")
plt.tight_layout()
plt.show()

## 11. Interpretation and limitations

The final interpretation considers the exploratory findings alongside cross-validation, validation, and test performance. Particular attention is given to the relationship between the validation ranking and final test results, as well as the balance between false-positive and false-negative predictions.

### Study limitations

- The Titanic dataset is a historical passenger sample and is not representative of a contemporary population.
- Relationships observed during exploratory analysis are descriptive and do not establish causation.
- The data is partitioned at the passenger-row level. Passengers belonging to the same family or sharing a ticket may therefore appear in different partitions; the evaluation does not establish group-independent generalization.
- The engineered features are intentionally limited to interpretable passenger-level transformations. More complex family- or ticket-level aggregation would require additional leakage-aware design.
- Performance estimates depend on the dataset version, partitioning seed, preprocessing choices, model configuration, and software environment.

These limitations define the scope within which the reported model comparisons should be interpreted.

In [ ]:
# Compact, reproducible summary for reporting
summary = {
    "selected_model": selected_name,
    "random_seed": SEED,
    "train_rows": len(X_train),
    "validation_rows": len(X_valid),
    "test_rows": len(X_test),
    **{f"test_{metric.lower().replace('-', '_')}": value
       for metric, value in test_metrics.items()}
}
display(pd.Series(summary, name="Value").to_frame())

## 12. Reproducibility and repository structure

The notebook is designed to run from the project repository with the labelled dataset supplied separately. The data directory is excluded from version control, while the notebook, dependency list, and project documentation form the reproducible project record.

```text
titanic-survival-analysis/
├── README.md
├── requirements.txt
├── .gitignore
├── notebooks/
│   └── titanic_survival_analysis.ipynb
└── data/
    └── raw/
        └── train.csv   # supplied locally; not committed
```

### Runtime dependencies

```text
numpy
pandas
matplotlib
seaborn
scikit-learn
jupyter
```

### Excluded local and generated files

```text
data/
.ipynb_checkpoints/
__pycache__/
*.py[cod]
.DS_Store
```

The experimental design keeps the training, validation, and test roles distinct: cross-validation and hyperparameter tuning use the training partition, validation performance determines model selection, and the held-out test partition is reserved for final evaluation.